In [0]:
data = [(1, "Alice", "IT", 29, 55000),
        (2, "Ben", "HR", 34, 48000),
        (3, "Cara", "IT", 41, 62000),
        (4, "Dev", "Finance", 26, 45000),
        (5, "Ella", None, 31, 51000)]
columns = ["id", "name", "department", "age", "salary"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
# Adding a New Column to a Spark DataFrame "salary_in_thousands"
#Note: withColumn() always returns a new DataFrame — it does not modify df in place. That's why you reassign it back to df (or a new variable name). Forgetting to reassign is a common early mistake — you'd run the transformation but never actually keep the result.

from pyspark.sql.functions import col
df = df.withColumn("salary_in_thousands", col("salary") / 1000)
df.display()

In [0]:
#Replace an Existing Column
df.withColumn("salary", col("salary").cast("double"))
df.printSchema()
df.display()

#Casting age column to Integer
df.withColumn("age", col("age").cast("int")).printSchema()
df.display()


In [0]:
#Common withColumn Patterns for Data Quality Work
# Flagging rows (a very common DQ Pattern)

# when()...otherwise() is PySpark's equivalent of a CASE WHEN in SQL. This pattern — flag a row as pass/fail based on a condition — is exactly how many data quality checks get built starting Phase 2.

from pyspark.sql.functions import when


df = df.withColumn(
    "high_earner",
    when(col("salary") > 50000, "Yes").otherwise("No")
)
df.display()


In [0]:
# Combining columns into a new one

from pyspark.sql.functions import concat, lit, col
df = df.withColumn("Full Label", concat(col("name"), lit(" - "), col("department")))
df.display()

#lit() wraps a plain literal value (like the string " - ") so it can be combined with column expressions — this trips people up at first, since you can't just write a raw string alongside a column.

In [0]:
# Type Casting
df = df.withColumn("age", col("age").cast("int"))
df.display()

#Note: You can also use PySpark's built-in cast() function to do the same thing. It's a little more verbose, but it's a little more flexible. For example, you can use it to cast a column to a different type based on a condition:
# Add df = to reassign the result
df = df.withColumn("age_category", 
    when(col("age") > 30, "Senior").otherwise("Junior"))
df.display()
                   

In [0]:


# Chaining multiple withColumn calls
df = (df
    .withColumn("salary", col("salary").cast("double"))
    .withColumn("salary_in_thousands", col("salary") / 1000)
    .withColumn("high_earner", when(col("salary") > 50000, "Yes").otherwise("No"))
)
df.display()


#⚠️ Chaining many withColumn() calls (dozens or more) can slow Spark down internally, because each call adds to the query plan. For a handful of columns like this it's completely fine — just something to be aware of once you reach Day 28 (Performance basics).

In [0]:
# Employee Table with 10 records
employee_data = [
    (1, "Alice Johnson", "IT", 29, 55000),
    (2, "Ben Smith", "HR", 34, 48000),
    (3, "Cara Martinez", "IT", 41, 62000),
    (4, "Dev Patel", "Finance", 26, 45000),
    (5, "Ella Brown", "Marketing", 31, 51000),
    (6, "Frank Wilson", "IT", 38, 68000),
    (7, "Grace Lee", "Finance", 27, 47000),
    (8, "Henry Davis", "HR", 44, 53000),
    (9, "Ivy Chen", "IT", 33, 59000),
    (10, "Jack Taylor", "Marketing", 29, 46000)
]

employee_columns = ["id", "name", "department", "age", "salary"]
employees = spark.createDataFrame(employee_data, employee_columns)

print(f"Total employees: {employees.count()}")
employees.display()

In [0]:

## Chaining all transformations together in a single statement
from pyspark.sql.functions import col, when, concat, lit

employees = (employees
    .withColumn("salary", col("salary").cast("double"))
    .withColumn("age", col("age").cast("int"))
    .withColumn("salary_in_thousands", col("salary") / 1000)
    .withColumn("high_earner", when(col("salary") > 50000, "Yes").otherwise("No"))
    .withColumn("age_category", when(col("age") > 30, "Senior").otherwise("Junior"))
    .withColumn("Full Label", concat(col("name"), lit(" - "), col("department")))
)

employees.display()